# Interactive Spectrum Analysis Template

**Experiment**: Radiation Detection with Radiacode 103 + Am-241

This enhanced notebook includes:
- Loading real Radiacode spectra using the dedicated parser
- ROI selection and net count rate calculation
- Gaussian peak fitting on the 59.5 keV Am-241 photopeak
- Interactive widgets for exploring fits in real time

## 1. Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.signal import find_peaks
import ipywidgets as widgets
from IPython.display import display
from pathlib import Path

# Import our custom Radiacode parser
import sys
sys.path.append('..')  # adjust if running from different location
from analysis.radiacode_parser import load_radiacode_spectrum, compute_roi_count_rate

plt.style.use('seaborn-v0_8-whitegrid')
print('All libraries and parser loaded successfully')

## 2. Load a Spectrum

Replace the example path with your actual Radiacode export file.

In [ ]:
# === USER INPUT ===
spectrum_file = 'data/raw/distance_10cm_300s_20260520_1420.csv'  # <-- change this
live_time = 300.0  # seconds
background_cps = 0.8

try:
    energy, counts, meta = load_radiacode_spectrum(spectrum_file)
    print('Loaded spectrum:', meta['filename'])
    print('Total counts:', meta['total_counts'])
    print('Energy range: {:.1f} - {:.1f} keV'.format(energy.min(), energy.max()))
except Exception as e:
    print('Could not load file. Using simulated data instead.')
    print(e)
    # Fallback simulated Am-241 like spectrum
    energy = np.linspace(0, 200, 1024)
    counts = np.random.poisson(2 + 180 * np.exp(-((energy - 59.5)**2) / (2 * 2.5**2)))
    meta = {'filename': 'simulated'} 

df = pd.DataFrame({'energy_keV': energy, 'counts': counts})
df.head()

## 3. Quick ROI Count Rate

In [ ]:
roi = (58.0, 62.0)  # typical for Am-241 59.54 keV
net_cps = compute_roi_count_rate(energy, counts, roi=roi, live_time=live_time, background_cps=background_cps)
print(f'Net count rate in ROI {roi}: {net_cps:.2f} cps')

## 4. Interactive Peak Fitting with Widgets

Adjust the ROI sliders and see the Gaussian fit update live.

In [ ]:
def gaussian(x, amp, mu, sigma, bg):
    return amp * np.exp(-((x - mu)**2) / (2 * sigma**2)) + bg

def fit_and_plot(roi_low=58.0, roi_high=62.0):
    mask = (energy >= roi_low) & (energy <= roi_high)
    x = energy[mask]
    y = counts[mask]

    if len(x) < 5:
        print('Not enough points in ROI')
        return

    # Initial guess
    amp0 = y.max() - y.min()
    mu0 = x[np.argmax(y)]
    sigma0 = 2.0
    bg0 = y.min()

    try:
        popt, pcov = curve_fit(gaussian, x, y, p0=[amp0, mu0, sigma0, bg0])
        amp, mu, sigma, bg = popt
        fit_y = gaussian(x, *popt)

        # Plot
        plt.figure(figsize=(10, 6))
        plt.plot(energy, counts, alpha=0.6, label='Full spectrum')
        plt.plot(x, y, 'o', label='ROI data')
        plt.plot(x, fit_y, 'r-', linewidth=2, label=f'Gaussian fit
μ={mu:.2f} keV, σ={sigma:.2f} keV')
        plt.axvline(mu, color='red', linestyle='--', alpha=0.7)
        plt.xlabel('Energy (keV)')
        plt.ylabel('Counts')
        plt.title(f'Peak Fit - ROI [{roi_low:.1f} - {roi_high:.1f}] keV')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.xlim(50, 70)
        plt.show()

        print(f'Peak position: {mu:.2f} ± {np.sqrt(pcov[1,1]):.2f} keV')
        print(f'FWHM: {2.355 * sigma:.2f} keV')
        print(f'Net area under peak (approx): {amp * sigma * np.sqrt(2*np.pi):.1f} counts')
    except Exception as e:
        print('Fit failed:', e)

# Create interactive widgets
roi_low_slider = widgets.FloatSlider(value=58.0, min=50.0, max=65.0, step=0.2, description='ROI Low (keV):')
roi_high_slider = widgets.FloatSlider(value=62.0, min=55.0, max=70.0, step=0.2, description='ROI High (keV):')

ui = widgets.interactive(fit_and_plot, roi_low=roi_low_slider, roi_high=roi_high_slider)
display(ui)

## 5. Batch Processing Multiple Spectra (Example)

Loop over several files and extract net CPS or peak parameters.

In [ ]:
# Example: process all CSV files in data/raw/
# raw_files = list(Path('data/raw').glob('*.csv'))
# results = []
# for f in raw_files:
#     e, c, m = load_radiacode_spectrum(f)
#     cps = compute_roi_count_rate(e, c, roi=(58,62), live_time=300)
#     results.append({'file': m['filename'], 'net_cps': cps})
# 
# results_df = pd.DataFrame(results)
# results_df

## 6. Next Steps & Customization Ideas

- Add energy calibration correction if your Radiacode needs it
- Implement better background subtraction (linear or polynomial fit outside ROI)
- Save fitted parameters to a results CSV for later plotting vs distance/shielding
- Add FWHM vs distance analysis
- Export high-quality figures for your protocols or thesis
- Connect to your Obsidian vault or Notion for automatic logging